# nbk_covid_BZ_to_SZ
## Bronze → Silver Transformation

**Purpose:** Read raw COVID-19 data from the Bronze layer, clean and standardize it, and write the result to the Silver layer.

**Input:** `Bronze/covid_worldwide_rd.csv`  
**Output:** `Silver/covid_worldwide_SZ.csv`

### Transformations applied
- Drop `Serial Number` column (not needed for analysis)
- Remove thousand-separator commas from numeric columns and cast to `float`
- Fill missing `Population` values manually for 3 known countries
- Fill missing `Total Deaths` values with `0`
- Drop remaining rows with nulls
- Export clean dataset to Silver layer

> ⚠️ **Before running:** update the ABFSS paths to match your own storage account name.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

In [ ]:
# ⚠️ Update storage account name if different
covid_raw = pd.read_csv(
    'abfss://datatokioolympicsgps@tokioolympicsgps.dfs.core.windows.net/Bronze/covid_worldwide_rd.csv',
    index_col=False
)
covid_raw.head()

In [ ]:
covid_raw.shape

In [ ]:
covid_raw.info()

In [ ]:
# Drop Serial Number — not needed for analysis
covid_raw.drop('Serial Number', axis=1, inplace=True)

In [ ]:
covid_raw.info()

In [ ]:
# Check null percentage per column
covid_raw.isnull().sum() / covid_raw.shape[0] * 100

In [ ]:
# Remove thousand-separator commas and cast numeric columns to float
covid_raw['Total Cases']     = covid_raw['Total Cases'].str.replace(',', '', regex=True).astype('float')
covid_raw['Total Deaths']    = covid_raw['Total Deaths'].str.replace(',', '', regex=True).astype('float')
covid_raw['Total Recovered'] = covid_raw['Total Recovered'].str.replace(',', '', regex=True).astype('float')
covid_raw['Active Cases']    = covid_raw['Active Cases'].str.replace(',', '', regex=True).astype('float')
covid_raw['Total Test']      = covid_raw['Total Test'].str.replace(',', '', regex=True).astype('float')
covid_raw['Population']      = covid_raw['Population'].str.replace(',', '', regex=True).astype('float')

In [ ]:
covid_raw.info()

In [ ]:
# Inspect rows with missing Population
covid_raw[covid_raw['Population'].isnull()]

In [ ]:
# Manually fill known population values (India, Niue, Nauru)
covid_raw.loc[90,  'Population'] = 1439323776
covid_raw.loc[226, 'Population'] = 2666
covid_raw.loc[229, 'Population'] = 2047

In [ ]:
# Recheck null percentage
covid_raw.isnull().sum() / covid_raw.shape[0] * 100

In [ ]:
# Inspect rows with missing Total Deaths
covid_raw[covid_raw['Total Deaths'].isnull()]

In [ ]:
# Fill missing Total Deaths with 0 (no recorded deaths)
covid_raw['Total Deaths'].fillna(0, inplace=True)

In [ ]:
# Drop any remaining rows with nulls
covid_raw.dropna(inplace=True)
covid_raw.info()

In [ ]:
covid_raw.Country.describe()

In [ ]:
# Export clean dataset to Silver layer
# ⚠️ Update storage account name if different
csv_file_path = 'abfss://datatokioolympicsgps@tokioolympicsgps.dfs.core.windows.net/Silver/covid_worldwide_SZ.csv'

covid_raw.to_csv(csv_file_path, index=False)
print(f'Dataframe exported to {csv_file_path}')